In [2]:
import pandas as pd
import numpy  as np

In [3]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [4]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    print(f"GPU is enabled. Using {torch.cuda.get_device_name(0)}.")
else:
    print("GPU is not enabled. Using CPU.")


GPU is enabled. Using Tesla T4.


In [5]:
from transformers import BertTokenizer, BertModel, BertConfig
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split


2025-05-01 08:06:54.202230: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746086814.500770      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746086814.580949      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
model = BertForSequenceClassification.from_pretrained('neuralmind/bert-base-portuguese-cased',
        num_labels=2 # Binary classification
)
model = model.to(device)

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [7]:
# Load the training data
train_df = pd.read_csv('/kaggle/input/tupy-e/binary_train.csv')
train_df = train_df[['text', 'hate','aggressive']].dropna()

# Load the test data
test_df = pd.read_csv('/kaggle/input/tupy-e/binary_test.csv')
test_df = test_df[['text', 'hate','aggressive']].dropna()


In [8]:
import re
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()  # Lowercase
    text = re.sub(r'@\w+', '', text)  # Remove @mentions
    text = re.sub(r'http\S+|www.\S+', '', text)  # Remove URLs
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    return text.strip()
train_df['text'] = train_df['text'].apply(preprocess_text)
test_df['text'] = test_df['text'].apply(preprocess_text)

In [9]:
label = np.zeros((len(train_df['text']), 2))
for i in range(len(train_df)):
    if train_df['hate'][i] == 0:
        label[i] = [1, 0]
    else:
        label[i] = [0, 1]


In [10]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    label.tolist(),
    test_size=0.2,
    random_state=42
)


In [11]:
# For training and validation
from datasets import load_dataset, Dataset
train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_labels}) # Changed 'hate' to 'labels'
val_dataset = Dataset.from_dict({'text': val_texts, 'labels': val_labels}) # Changed 'hate' to 'labels'

# For testing
test_labels = [[1, 0] if label == 0 else [0, 1] for label in test_df['hate'].tolist()]
test_dataset = Dataset.from_dict({'text': test_df['text'].tolist(), 'labels': test_labels}) # Changed 'hate' to 'labels'

# Tokenize all
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/27947 [00:00<?, ? examples/s]

Map:   0%|          | 0/6987 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

In [ ]:


training_args = TrainingArguments(
    output_dir='./results',  # Directory to save the model checkpoints
    eval_strategy="epoch",  # Evaluate after each epoch
    
    logging_strategy="no",  # Disables logging
    save_strategy="epoch",  # Save model checkpoint after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    num_train_epochs=20,  # Number of epochs to train
    per_device_train_batch_size=16,  # Batch size per device for training
    per_device_eval_batch_size=16,  # Batch size per device for evaluation
    learning_rate=2e-5,  # Learning rate for optimization
    weight_decay=0.01,  # Weight decay (for regularization)
    logging_dir='./logs',  # Directory to save logs
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model='accuracy',  # Metric to track the best model
    report_to=[],  # Don't report metrics to any logging service
)


In [12]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Check if labels are multilabel-indicator (2D array)
    if labels.ndim == 2 and labels.shape[1] > 1:
        # If multilabel-indicator, convert to binary format
        labels = labels.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


NameError: name 'training_args' is not defined

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./my-bertimbau-hate-model")
tokenizer.save_pretrained("./my-bertimbau-hate-model")


In [14]:
def preprocess_data(example):
    example['labels'] = torch.tensor(example['labels'], dtype=torch.long)
    return example
test_dataset = test_dataset.map(preprocess_data)


Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

/tmp/ipykernel_31/1548902723.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  example['labels'] = torch.tensor(example['labels'], dtype=torch.long)


In [15]:

model1 = BertForSequenceClassification.from_pretrained("/kaggle/input/tupy-e-bert")
tokenizer = BertTokenizer.from_pretrained("/kaggle/input/tupy-e-bert")

In [16]:
cleaned_test_dataset = Dataset.from_dict({'text': test_df['text'].tolist()})
cleaned_test_dataset = cleaned_test_dataset.map(tokenize, batched=True)
cleaned_test_dataset.set_format('torch', columns=['input_ids', 'attention_mask'])

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

In [17]:
trainer_2 = Trainer(model=model1,)

In [18]:
probabilities=[]

In [19]:
from tqdm import tqdm
predicted_classes = []

for i in tqdm(range(len(test_dataset['text']))):
    text = test_dataset['text'][i]
    
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}
    
    with torch.no_grad():
        outputs = model1(**inputs)
        logits = outputs.logits

        predicted_class = torch.argmax(logits, dim=-1).item()  # Get class index
        predicted_classes.append(predicted_class)


100%|██████████| 8734/8734 [02:34<00:00, 56.52it/s]


In [20]:
print(test_dataset['labels'])

tensor([[1, 0],
        [1, 0],
        [1, 0],
        ...,
        [1, 0],
        [0, 1],
        [1, 0]])


In [21]:
predictions =[]
for i in tqdm(range(len(probabilities))):
    x =  probabilities[i]
    if(x[0][0]>x[0][1]): predictions.append(0)
    else: predictions.append(1)



0it [00:00, ?it/s]


In [22]:
print(len(test_dataset['labels']),len(predicted_classes))

8734 8734


In [23]:
import torch
from sklearn.metrics import accuracy_score,classification_report

# Convert one-hot encoded true labels to class indices
true_labels = torch.tensor(test_dataset['labels']).argmax(dim=1).tolist()

# Ensure predicted_classes is also a list of class indices
accuracy = accuracy_score(true_labels, predicted_classes)
print(classification_report(true_labels,predicted_classes))
print(f"Accuracy: {accuracy:.4f}")


              precision    recall  f1-score   support

           0       0.92      0.98      0.95      7683
           1       0.70      0.39      0.50      1051

    accuracy                           0.91      8734
   macro avg       0.81      0.68      0.72      8734
weighted avg       0.89      0.91      0.89      8734

Accuracy: 0.9060


/tmp/ipykernel_31/223260750.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  true_labels = torch.tensor(test_dataset['labels']).argmax(dim=1).tolist()


In [24]:
test_dataset['labels'][0]

tensor([1, 0])